# All VLM Stages — One-Shot Benchmark Harness (any model, one notebook)

**One parametrized notebook, not one notebook per model.** Set `MODEL` in the config cell,
Run All, and every VLM stage is benchmarked in order. Each run appends rows to a shared
results CSV (pushed to the private HF dataset repo), so cross-model comparison accumulates
across runs of this same notebook. To test one model on one stage only: set `MODEL`, run
cells 1-5, then run just that stage's section.

**Supported models** (adapter layer, cell 5): `gpt-5.5` (OpenAI API, low reasoning —
the incumbent per current production info), `qwen-base` (raw Qwen3-VL-8B), `qwen-domain-v1`
/ `qwen-domain-v2` (our trained adapters from HF), `molmo2` (Molmo2-O-7B).

**What each stage's score honestly means — read this before quoting numbers:**

| Stage | Ground truth | Score meaning |
|---|---|---|
| 1 Sheet classification | ❌ none (all our pages are P&IDs) | Recall on known-P&ID pages ONLY — a model that says "P&ID" to everything scores 100%. One-sided, flagged. |
| 2 Title block | ❌ none | Raw extractions stored per model. Score = cross-model agreement, computed when ≥2 models have rows in the CSV. No absolute truth. |
| 5 OCR tiebreaker | ⚠️ synthesized | Real accuracy on a controlled task: true word vs 1-char-corrupted word, given the crop. Pseudo-GT from Tesseract-confident words. |
| 10.5 Skid grouping | ⚠️ sparse | **Not benchmarked yet** — PID2Graph grouping labels too thin. Stage 12's relation score is the closest proxy. |
| 12 Relation validation | ✅ real (PID2Graph) | Accept/reject accuracy on injected true/false connections. The most trustworthy number here. |
| 13 Entity validation | ⚠️ partial (Gupta, class-agnostic) | Keep/remove accuracy: real symbol crops vs empty-region decoys. Tests existence-validation, not type-correctness. |

Stage 4 (detection) is deliberately NOT here — it has its own dedicated harness/notebooks.

**Cost note (gpt-5.5):** ~150 image calls per full run at default N. Watch your usage if you
crank `N_PER_STAGE` up.

## 1. Config — pick the model, paste keys

In [ ]:
MODEL = "gpt-5.5"   # one of: gpt-5.5 | qwen-base | qwen-domain-v1 | qwen-domain-v2 | molmo2

HF_TOKEN = "paste-your-hf-token-here"          # needed for data + results push (all models)
OPENAI_API_KEY = "paste-your-openai-key-here"  # only needed when MODEL = "gpt-5.5"

DATA_REPO = "timthy45/pnid-extraction-datasets"
CKPT_REPO = "timthy45/qwen3vl-pnid-domain-base"
RESULTS_PATH_IN_REPO = "benchmarks/all_vlm_stages_results.csv"

N_PER_STAGE = 25          # examples per stage (stage 1 capped by the 20 test sheets)
GPT_REASONING_EFFORT = "low"   # matches reported production config

assert HF_TOKEN.startswith("hf_"), "paste HF token"
if MODEL == "gpt-5.5":
    assert OPENAI_API_KEY.startswith("sk-"), "paste OpenAI key for gpt-5.5"

## 2. Install + data from HF (Gupta + PID2Graph; local disk, no Drive)

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip install -q pytesseract huggingface_hub openai peft
!pip uninstall -y torchao -q

import zipfile, time, json, random, re
from pathlib import Path
from huggingface_hub import hf_hub_download

DATA = Path("/content/data")
DATA.mkdir(exist_ok=True)
t0 = time.time()
for fname, extract_to in [
    ("gupta_pid/PID_Dataset.zip", DATA / "gupta"),
    ("pid2graph/PID2Graph.zip", DATA / "pid2graph"),
]:
    if extract_to.exists() and any(extract_to.iterdir()):
        print(f"{extract_to} cached"); continue
    zp = hf_hub_download(repo_id=DATA_REPO, filename=fname, repo_type="dataset", token=HF_TOKEN)
    extract_to.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(extract_to)
    print(f"extracted {fname} ({time.time()-t0:.0f}s)")

GUPTA_RAW = DATA / "gupta" / "PID_Dataset" / "0__raw_data"
PID2GRAPH_OPEN100 = DATA / "pid2graph" / "PID2Graph" / "Patched" / "PID2Graph OPEN100"
assert (GUPTA_RAW / "sheets" / "test").exists() and PID2GRAPH_OPEN100.exists()
print("data ready")

## 3. Shared helpers

Frozen 20-sheet Gupta test split for stages 1/2/13 (test sheets, never train), PID2Graph
for stage 12, Tesseract-confident words for stage 5.

In [ ]:
import xml.etree.ElementTree as ET
import pytesseract
from PIL import Image

Image.MAX_IMAGE_PIXELS = None
rng = random.Random(42)

TEST_SHEETS = sorted((GUPTA_RAW / "sheets" / "test").glob("*.*"))
TEST_LABELS = GUPTA_RAW / "labels" / "test"
print(f"{len(TEST_SHEETS)} frozen test sheets")

def yolo_boxes(label_path, W, H):
    boxes = []
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            if line.strip():
                parts = line.split()
                cx, cy, w, h = (float(v) for v in parts[1:5])
                boxes.append([ (cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H ])
    return boxes

def parse_graphml(path):
    root = ET.parse(path).getroot()
    ns = {"g": "http://graphml.graphdrawing.org/xmlns"}
    keymap = {k.get("id"): k.get("attr.name") for k in root.findall("g:key", ns)}
    nodes, edges = {}, set()
    for node in root.iter("{http://graphml.graphdrawing.org/xmlns}node"):
        vals = {keymap.get(d.get("key"), ""): d.text for d in node.findall("g:data", ns)}
        try:
            nodes[node.get("id")] = [float(vals["xmin"]), float(vals["ymin"]),
                                     float(vals["xmax"]), float(vals["ymax"])]
        except (KeyError, TypeError, ValueError):
            continue
    for e in root.iter("{http://graphml.graphdrawing.org/xmlns}edge"):
        if e.get("source") in nodes and e.get("target") in nodes:
            edges.add(frozenset((e.get("source"), e.get("target"))))
    return nodes, edges

## 4. Model adapter layer

Every model reduces to one function: `generate(pil_image, prompt) -> text`. This is what
makes the harness model-agnostic — the stage sections below never know which model runs.

In [ ]:
import base64, io

def _load_gpt55():
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    def generate(image, prompt, max_tokens=256):
        buf = io.BytesIO()
        image.save(buf, format="PNG")
        b64 = base64.standard_b64encode(buf.getvalue()).decode()
        resp = client.chat.completions.create(
            model="gpt-5.5",
            reasoning_effort=GPT_REASONING_EFFORT,
            max_completion_tokens=max_tokens,
            messages=[{"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                {"type": "text", "text": prompt},
            ]}],
        )
        return (resp.choices[0].message.content or "").strip()
    return generate

def _load_qwen(adapter_subdir=None):
    import torch
    from transformers import AutoModelForImageTextToText, AutoProcessor
    processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-8B-Instruct")
    model = AutoModelForImageTextToText.from_pretrained(
        "Qwen/Qwen3-VL-8B-Instruct", dtype=torch.bfloat16, device_map="cuda")
    if adapter_subdir:
        from huggingface_hub import snapshot_download
        from peft import PeftModel
        local = Path("/content/bench_ckpt")
        snapshot_download(repo_id=CKPT_REPO, repo_type="model", token=HF_TOKEN,
                          allow_patterns=[f"{adapter_subdir}/*"], local_dir=str(local))
        model = PeftModel.from_pretrained(model, str(local / adapter_subdir))
    model.eval()
    def generate(image, prompt, max_tokens=256):
        messages = [{"role": "user", "content": [{"type": "image", "image": image},
                                                 {"type": "text", "text": prompt}]}]
        inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                               return_dict=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
        return processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return generate

def _load_molmo2():
    import torch
    from transformers import AutoModelForImageTextToText, AutoProcessor
    mid = "allenai/Molmo2-O-7B"
    processor = AutoProcessor.from_pretrained(mid, trust_remote_code=True)
    model = AutoModelForImageTextToText.from_pretrained(
        mid, trust_remote_code=True, dtype=torch.bfloat16, device_map="cuda").eval()
    def generate(image, prompt, max_tokens=256):
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt},
                                                 {"type": "image", "image": image}]}]
        inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                               return_dict=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
        return processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return generate

LOADERS = {
    "gpt-5.5": _load_gpt55,
    "qwen-base": lambda: _load_qwen(None),
    "qwen-domain-v1": lambda: _load_qwen("latest"),
    "qwen-domain-v2": lambda: _load_qwen("v2/latest"),
    "molmo2": _load_molmo2,
}
assert MODEL in LOADERS, f"unknown MODEL {MODEL}"
generate = LOADERS[MODEL]()
print(f"model ready: {MODEL}")

## 5. Results plumbing — every stage appends here; CSV pushed to HF at the end

In [ ]:
import csv, datetime

RESULTS_LOCAL = Path("/content/all_vlm_stages_results.csv")
FIELDS = ["timestamp", "model", "stage", "metric", "value", "n", "notes"]

# pull existing results so runs accumulate across models
try:
    prev = hf_hub_download(repo_id=DATA_REPO, filename=RESULTS_PATH_IN_REPO,
                           repo_type="dataset", token=HF_TOKEN)
    RESULTS_LOCAL.write_text(Path(prev).read_text())
    print("pulled existing results CSV")
except Exception:
    with open(RESULTS_LOCAL, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=FIELDS).writeheader()
    print("started fresh results CSV")

def record(stage, metric, value, n, notes=""):
    with open(RESULTS_LOCAL, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=FIELDS).writerow({
            "timestamp": datetime.datetime.utcnow().isoformat(timespec="seconds"),
            "model": MODEL, "stage": stage, "metric": metric,
            "value": round(value, 4) if isinstance(value, float) else value,
            "n": n, "notes": notes})
    print(f"  [recorded] {stage} {metric}={value} (n={n})")

## 6. Stage 1 — Sheet Classification  ⚠️ recall-only

All 20 frozen test sheets are P&IDs; we have no labeled legend/notes/cover pages. So this
measures ONLY "does the model recognize a P&ID as a P&ID" — a yes-to-everything model scores
100%. Treat as a smoke check, not a discriminative benchmark, until non-P&ID pages exist.

In [ ]:
STAGE1_PROMPT = ("Classify this engineering document page as exactly one of: "
                 "pid_drawing, legend, notes, cover. A pid_drawing shows process equipment "
                 "connected by lines. Answer with just the label.")

correct = 0
for sheet in TEST_SHEETS[:N_PER_STAGE]:
    img = Image.open(sheet).convert("RGB")
    img.thumbnail((1568, 1568))
    ans = generate(img, STAGE1_PROMPT, max_tokens=16).lower()
    ok = "pid" in ans
    correct += ok
    print(f"{sheet.stem:6s} -> {ans[:40]:42s} {'OK' if ok else 'MISS'}")

n = min(len(TEST_SHEETS), N_PER_STAGE)
record("1_sheet_classification", "pid_recall_%", 100 * correct / n, n,
       "one-sided: all eval pages are P&IDs, no negative class available")

## 7. Stage 2 — Title Block extraction  ⚠️ agreement-only

No ground-truth metadata labels exist. Each model's raw extraction is stored; comparing rows
across models in the CSV gives cross-model agreement. A single run's output is unscoreable.

In [ ]:
STAGE2_PROMPT = ("This is the bottom-right corner of an engineering drawing containing the "
                 "title block. Extract: drawing number, revision, and title. "
                 "Answer as: number=<...> | rev=<...> | title=<...>")

extractions = []
for sheet in TEST_SHEETS[:min(10, N_PER_STAGE)]:
    img = Image.open(sheet).convert("RGB")
    W, H = img.size
    crop = img.crop((int(W * 0.55), int(H * 0.70), W, H))   # bottom-right corner
    ans = generate(crop, STAGE2_PROMPT, max_tokens=128)
    extractions.append(ans.replace("\n", " ")[:200])
    print(f"{sheet.stem:6s} -> {extractions[-1][:100]}")

for sheet, ext in zip(TEST_SHEETS, extractions):
    record("2_title_block", f"extraction_{sheet.stem}", ext, 1,
           "raw extraction - agreement scored across models, no absolute GT")

## 8. Stage 5 — OCR tiebreaker  (synthesized, scored)

The real stage adjudicates between OCR candidates on low-confidence words. Controlled proxy:
a tag crop + the true word vs a 1-character-corrupted version, model picks A or B.
Pseudo-GT from Tesseract-confident (≥70) words on test sheets.

In [ ]:
def corrupt(word):
    i = rng.randrange(len(word))
    repl = {"0": "8", "1": "7", "5": "6", "B": "8", "O": "0", "I": "1"}.get(
        word[i], rng.choice("0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ-"))
    while repl == word[i]:
        repl = rng.choice("0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ-")
    return word[:i] + repl + word[i+1:]

pairs = []
for sheet in TEST_SHEETS:
    if len(pairs) >= N_PER_STAGE:
        break
    img = Image.open(sheet).convert("RGB")
    W, H = img.size
    tile = img.crop((0, 0, min(2048, W), min(2048, H)))
    data = pytesseract.image_to_data(tile, output_type=pytesseract.Output.DICT)
    for i in range(len(data["text"])):
        t = data["text"][i].strip()
        if (len(t) >= 4 and int(data["conf"][i]) >= 70
                and (any(c.isdigit() for c in t) or "-" in t) and len(pairs) < N_PER_STAGE):
            m = 20
            crop = tile.crop((max(0, data["left"][i]-m), max(0, data["top"][i]-m),
                              data["left"][i]+data["width"][i]+m, data["top"][i]+data["height"][i]+m))
            pairs.append({"crop": crop, "true": t, "fake": corrupt(t)})

correct = undecided = 0
for p in pairs:
    a_is_true = rng.random() < 0.5
    A, B = (p["true"], p["fake"]) if a_is_true else (p["fake"], p["true"])
    prompt = (f"This crop shows one text label from an engineering drawing. Two OCR readings "
              f"were proposed:\nA: {A}\nB: {B}\nWhich exactly matches the pixels? Answer A or B only.")
    ans = generate(p["crop"], prompt, max_tokens=8).strip().upper()
    if not ans or ans[0] not in "AB":
        undecided += 1
        continue
    correct += (ans[0] == "A") == a_is_true

n_decided = len(pairs) - undecided
record("5_ocr_tiebreaker", "accuracy_%", 100 * correct / n_decided if n_decided else 0.0,
       len(pairs), f"pseudo-GT from Tesseract conf>=70; undecided={undecided}; baseline 50%")
print(f"accuracy {100*correct/max(n_decided,1):.1f}% on {n_decided} decided ({undecided} undecided)")

## 9. Stage 12 — Relation Validation  ✅ real ground truth

The stage's real job: confirm/reject proposed connections. Injected 50/50 true/false pairs
from PID2Graph OPEN100 graph edges. The most trustworthy score in this harness.

In [ ]:
MAX_SPAN, MARGIN = 1400, 80
rel_examples = []
gmls = sorted(PID2GRAPH_OPEN100.rglob("*.graphml"))
rng.shuffle(gmls)
for gml in gmls:
    if len(rel_examples) >= N_PER_STAGE:
        break
    png = gml.with_suffix(".png")
    if not png.exists():
        continue
    try:
        nodes, edges = parse_graphml(gml)
    except ET.ParseError:
        continue
    if len(nodes) < 4 or not edges:
        continue
    want_connected = len(rel_examples) % 2 == 0
    pair = None
    if want_connected:
        pool = [tuple(e) for e in edges]
        pair = pool[rng.randrange(len(pool))]
    else:
        for _ in range(50):
            a, b = rng.sample(list(nodes), 2)
            if frozenset((a, b)) not in edges:
                pair = (a, b); break
    if not pair:
        continue
    a, b = pair
    ba, bb = nodes[a], nodes[b]
    ux0, uy0 = min(ba[0], bb[0]), min(ba[1], bb[1])
    ux1, uy1 = max(ba[2], bb[2]), max(ba[3], bb[3])
    if ux1 - ux0 > MAX_SPAN or uy1 - uy0 > MAX_SPAN:
        continue
    cx0, cy0 = max(0, int(ux0 - MARGIN)), max(0, int(uy0 - MARGIN))
    img = Image.open(png).convert("RGB")
    crop = img.crop((cx0, cy0, int(ux1 + MARGIN), int(uy1 + MARGIN)))
    prompt = (f"In this P&ID crop there is a symbol at "
              f"[{int(ba[0]-cx0)}, {int(ba[1]-cy0)}, {int(ba[2]-cx0)}, {int(ba[3]-cy0)}] and another at "
              f"[{int(bb[0]-cx0)}, {int(bb[1]-cy0)}, {int(bb[2]-cx0)}, {int(bb[3]-cy0)}] "
              f"(pixel coordinates, top-left origin). Are these two symbols directly connected "
              f"to each other? Answer yes or no.")
    rel_examples.append({"crop": crop, "prompt": prompt, "connected": want_connected})

correct = undecided = 0
for ex in rel_examples:
    ans = generate(ex["crop"], ex["prompt"], max_tokens=16).lower()
    yes, no = ans.startswith("yes"), ans.startswith("no")
    if not yes and not no:
        undecided += 1; continue
    correct += (yes == ex["connected"])

n_decided = len(rel_examples) - undecided
record("12_relation_validation", "accuracy_%",
       100 * correct / n_decided if n_decided else 0.0, len(rel_examples),
       f"real PID2Graph GT, 50/50 balance; undecided={undecided}; baseline 50%")
print(f"accuracy {100*correct/max(n_decided,1):.1f}% on {n_decided} decided ({undecided} undecided)")

## 10. Stage 13 — Entity Validation  ⚠️ existence-only

Real job: keep/correct/remove each detected entity. Class-agnostic proxy with real Gupta
boxes: crop around a true symbol ("keep" expected) vs an empty region decoy ("remove"
expected). Tests existence-validation only — type-correctness is untestable without typed
real GT (the known permanent Stage-4 limitation applies here too).

In [ ]:
def overlaps_any(box, boxes):
    return any(box[0] < b[2] and box[2] > b[0] and box[1] < b[3] and box[3] > b[1] for b in boxes)

val_examples = []
for sheet in TEST_SHEETS:
    if len(val_examples) >= N_PER_STAGE:
        break
    img = Image.open(sheet).convert("RGB")
    W, H = img.size
    boxes = yolo_boxes(TEST_LABELS / f"{sheet.stem}.txt", W, H)
    if not boxes:
        continue
    want_real = len(val_examples) % 2 == 0
    if want_real:
        b = boxes[rng.randrange(len(boxes))]
    else:
        b = None
        for _ in range(80):
            size = rng.uniform(30, 80)
            x0 = rng.uniform(0, W - size); y0 = rng.uniform(0, H - size)
            cand = [x0, y0, x0 + size, y0 + size]
            if not overlaps_any(cand, boxes):
                b = cand; break
        if b is None:
            continue
    m = 60
    crop = img.crop((max(0, int(b[0]-m)), max(0, int(b[1]-m)),
                     min(W, int(b[2]+m)), min(H, int(b[3]+m))))
    cb = [int(b[0] - max(0, b[0]-m)), int(b[1] - max(0, b[1]-m)),
          int(b[2] - max(0, b[0]-m)), int(b[3] - max(0, b[1]-m))]
    prompt = (f"A symbol detector claims there is a P&ID equipment symbol at "
              f"[{cb[0]}, {cb[1]}, {cb[2]}, {cb[3]}] in this crop (pixel coords, top-left origin). "
              f"Verify against the pixels: is there really a discrete equipment symbol there "
              f"(valve, instrument, fitting)? Answer keep or remove.")
    val_examples.append({"crop": crop, "prompt": prompt, "is_real": want_real})

correct = undecided = 0
for ex in val_examples:
    ans = generate(ex["crop"], ex["prompt"], max_tokens=8).lower()
    keep, remove = ans.startswith("keep"), ans.startswith("remove")
    if not keep and not remove:
        undecided += 1; continue
    correct += (keep == ex["is_real"])

n_decided = len(val_examples) - undecided
record("13_entity_validation", "accuracy_%",
       100 * correct / n_decided if n_decided else 0.0, len(val_examples),
       f"existence-only (class-agnostic Gupta GT); undecided={undecided}; baseline 50%")
print(f"accuracy {100*correct/max(n_decided,1):.1f}% on {n_decided} decided ({undecided} undecided)")

## 11. Push results to HF + summary for this run

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.upload_file(path_or_fileobj=str(RESULTS_LOCAL), path_in_repo=RESULTS_PATH_IN_REPO,
                repo_id=DATA_REPO, repo_type="dataset")
print(f"results pushed to {DATA_REPO}/{RESULTS_PATH_IN_REPO}\n")

import csv as _csv
rows = list(_csv.DictReader(open(RESULTS_LOCAL)))
models = sorted({r["model"] for r in rows})
print(f"{'stage.metric':42s}" + "".join(f"{m:>16s}" for m in models))
print("-" * (42 + 16 * len(models)))
keys = sorted({(r["stage"], r["metric"]) for r in rows if not r["metric"].startswith("extraction_")})
for stage, metric in keys:
    line = f"{stage + '.' + metric:42s}"
    for m in models:
        vals = [r["value"] for r in rows if r["model"] == m and r["stage"] == stage and r["metric"] == metric]
        line += f"{vals[-1] if vals else '-':>16s}"
    print(line)
print("\nStage 2 extractions are stored as raw rows - compare across models manually or ask "
      "Claude to score agreement once 2+ models have run.")